#  using LangChain + Hugging Face

# 1. Install Required Libraries


# 2. Load Environment Variables

In [ ]:

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")


# 3. Load Data from Website

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.smith.langchain.com/tutorials/Administrators/manage_spend")
docs = loader.load()

print("Docs loaded:", len(docs))
print(docs[0].page_content[:500])


# 4. Split Text into Chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(docs)

print("Chunks:", len(documents))
print(documents[0].page_content[:500])


# 5. Create Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 6. Store Embeddings in FAISS
vectorstore = FAISS.from_documents(documents, embeddings)

#7. Create Retriever
retriever = vectorstore.as_retriever()


# 8. Load LLM

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"]
)


#  9. Create Prompt Template


prompt = ChatPromptTemplate.from_template("""
Answer ONLY using the context below:

<context>
{context}
</context>

Question: {input}
""")

# 10. Create Document Chain

document_chain = create_stuff_documents_chain(llm, prompt)


# 11. Create Retrieval Chain

In [ ]:
from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)


# 12. Ask Question

In [ ]:
response = retrieval_chain.invoke({
    "input": "LangSmith has two usage limits"
})

print("ANSWER:\n", response["answer"])


# 13. Print Final Answer

In [ ]:
print("\n--- Retrieved Context ---\n")

# 14. Print Retrieved Chunks
for i, doc in enumerate(response["context"][:3]):
    print(f"Chunk {i+1}:")
    print(doc.page_content[:300])
    print("-"*50)
